# EDA y Clasificación Titanic con RAPIDS
Usamos `cudf`, `cuml` y, cuando hace falta, `cupy` para mantener todo en GPU mientras exploramos y entrenamos modelos.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [ ]:
import cudf
import cupy as cp
import matplotlib.pyplot as plt
import seaborn as sns
from cuml.preprocessing import StandardScaler
from cuml.model_selection import train_test_split
from cuml.ensemble import RandomForestClassifier
from cuml.linear_model import LogisticRegression
from cuml import metrics
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
df = cudf.read_csv('Actividades/titanic.csv')
print('Columnas y Nulos por columna:')
print(df.isnull().sum())
print('Tamaño dataset', len(df))


In [ ]:
df = df.drop(columns=['Ticket'])
default_embarked = df['Embarked'].mode().iloc[0]
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna(default_embarked).str.upper()
df['Sex'] = df['Sex'].fillna('unknown').str.lower()
df['Name'] = df['Name'].fillna('Desconocido').str.title()
print('Nulos tras limpieza:')
print(df[['Age', 'Fare', 'Embarked']].isnull().sum())


In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = ((df['SibSp'] + df['Parch']) == 0).astype('int32')
print(df[['FamilySize','IsAlone']].head())


In [ ]:
feature_columns = ['Pclass','Age','SibSp','Parch','Fare','FamilySize','IsAlone']
cat_columns = ['Sex','Embarked']
df_numeric = df[feature_columns]
df_encoded = cudf.get_dummies(df[cat_columns])
df_ready = cudf.concat([df[['Survived']], df_numeric, df_encoded], axis=1)
print('Dimensiones finales:', df_ready.shape)


In [ ]:
scaler = StandardScaler()
cols_to_scale = df_ready.columns.drop('Survived')
scaled = scaler.fit_transform(df_ready[cols_to_scale])
scaled_df = cudf.DataFrame(scaled, columns=cols_to_scale)
df_scaled = cudf.concat([df_ready[['Survived']].reset_index(drop=True), scaled_df.reset_index(drop=True)], axis=1)
print(df_scaled.describe().loc[['mean','std']])


In [ ]:
X = df_scaled.drop(columns=['Survived'])
y = df_scaled['Survived']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=128, max_depth=6)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_val)[:, 1]  # cupy array
    preds = model.predict(X_val)
    print(f'{name} accuracy:', metrics.accuracy_score(y_val, preds))
    try:
        print(f'{name} ROC AUC:', metrics.roc_auc_score(y_val, probs))
    except ValueError:
        pass


In [ ]:
best_model = models['Random Forest']
probs_best = best_model.predict_proba(X_val)[:, 1].get()
fpr, tpr, _ = metrics.roc_curve(y_val.get(), probs_best)
precision, recall, _ = metrics.precision_recall_curve(y_val.get(), probs_best)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, label='ROC')
axes[0].set_title('ROC - Random Forest')
axes[1].plot(recall, precision, label='PR')
axes[1].set_title('Precisión-Recall - Random Forest')
plt.tight_layout()
plt.show()


In [ ]:
param_grid = {'n_estimators': [64, 128], 'max_depth': [6, 8]}
best_score = -cp.inf
best_params = {}
for n in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        model = RandomForestClassifier(n_estimators=n, max_depth=depth, random_state=0)
        model.fit(X_train, y_train)
        score = metrics.roc_auc_score(y_val, model.predict_proba(X_val)[:,1])
        if score > best_score:
            best_score = score
            best_params = {'n_estimators': n, 'max_depth': depth}
            tuned_model = model
print('Mejores parámetros paralelos CPU/GPU:', best_params)


In [ ]:
import joblib
joblib.dump(tuned_model, 'outputs/rapids_model.joblib')
print('Modelo RAPIDS guardado en outputs/rapids_model.joblib')


In [8]:
!pip install pycaret==3.3.2

In [9]:
# Classification Functional API Example

# loading sample dataset
from pycaret.datasets import get_data
data = get_data('juice')

# init setup
from pycaret.classification import *
s = setup(data, target = 'Purchase', session_id = 123)

# model training and selection
best = compare_models()

# evaluate trained model
evaluate_model(best)

# predict on hold-out/test set
pred_holdout = predict_model(best)

# predict on new data
new_data = data.copy().drop('Purchase', axis = 1)
predictions = predict_model(best, data = new_data)

# save model
save_model(best, 'best_pipeline')

RuntimeError: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')